In [24]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np
import torch

import wandb
from wandb.integration.xgboost import WandbCallback # Import the W&B callback


import os


In [ ]:

data_dir = '../../data/processed/eeg'

# load a single case for now
file_path = os.path.join(data_dir, 'case_36.pt')

# Load the data using PyTorch
# weights_only=False because we are loading a dictionary, not just model weights
loaded_data = torch.load(file_path, weights_only=False)

# Extract the variables based on the dictionary keys defined in preprocessing_3.py
features = loaded_data['features']
bis = loaded_data['bis']
caseid = loaded_data['caseid']

# Verify
print(f"Loaded Case ID: {caseid}")
print(f"Features tensor shape: {features.shape}")
print(f"BIS tensor shape: {bis.shape}")

Loaded Case ID: 36
Features tensor shape: torch.Size([4477, 19])
BIS tensor shape: torch.Size([4477])


In [15]:
import glob

# Get a list of all .pt files in the directory
all_files = glob.glob(os.path.join(data_dir, '*.pt'))

print(f"Found {len(all_files)} processed cases.")

# Example of looping through the first 5 files
for file in all_files:
    data = torch.load(file, weights_only=False)
    # print(f"Case {data['caseid']} has {data['features'].shape[0]} seconds of data.")

print(f"Keys: {list(data.keys())}")
print(f"Number of features: {data['features'].shape[1]}")
print(f"Features length: {data['features'].shape[0]}")

Found 406 processed cases.
Keys: ['features', 'bis', 'caseid']
Number of features: 19
Features length: 5974


In [ ]:
cases_master = pd.read_csv('../../data/processed/cases_data.csv')

ids = cases_master['caseid'].tolist()

INPUT_DIR = '../../data/processed/eeg'

# Extract and concatenate case files
dfs = []
ys = []
for id in ids:
    sample_path = os.path.join(INPUT_DIR, f'case_{id}.pt')
    sample_tensor = torch.load(sample_path)
    sample_df = pd.DataFrame(sample_tensor['features'].numpy())
    y = sample_tensor['bis'].numpy()

    sample_df['caseid'] = id
    dfs.append(sample_df)
    ys.append(y)

In [3]:
X = features.cpu().numpy() if features.is_cuda else features.numpy()
y = bis.cpu().numpy() if bis.is_cuda else bis.numpy()
print(f"Converted X shape: {X.shape}, y shape: {y.shape}")

Converted X shape: (4477, 19), y shape: (4477,)


In [4]:
# Chronological Train-Test Split (80% Train, 20% Test)
# Setting shuffle=False prevents data leakage by keeping the timeline intact
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [5]:
# Initialize the XGBoost Regressor
# These are standard starting hyperparameters; you can tune them later
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',  # Standard loss function for regression
    n_estimators=200,              # Number of boosting rounds/trees
    learning_rate=0.05,            # Step size shrinkage
    max_depth=6,                   # Maximum tree depth
    subsample=0.8,                 # Use 80% of data per tree to prevent overfitting
    colsample_bytree=0.8,          # Use 80% of features per tree
    random_state=42
)

In [6]:
# Train the Model
print("\nTraining XGBoost Regressor...")
xgb_model.fit(
    X_train, 
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)], # Monitor both sets to watch for overfitting
    verbose=20                                       # Print progress every 20 trees
)


Training XGBoost Regressor...
[0]	validation_0-rmse:13.00035	validation_1-rmse:19.28558
[20]	validation_0-rmse:6.86531	validation_1-rmse:14.85327
[40]	validation_0-rmse:4.55948	validation_1-rmse:14.99418
[60]	validation_0-rmse:3.67568	validation_1-rmse:15.33576
[80]	validation_0-rmse:3.30165	validation_1-rmse:15.58600
[100]	validation_0-rmse:3.05759	validation_1-rmse:15.60601
[120]	validation_0-rmse:2.84667	validation_1-rmse:15.69687
[140]	validation_0-rmse:2.64975	validation_1-rmse:15.74398
[160]	validation_0-rmse:2.49704	validation_1-rmse:15.78802
[180]	validation_0-rmse:2.33841	validation_1-rmse:15.80981
[199]	validation_0-rmse:2.20741	validation_1-rmse:15.83449


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [7]:
# Evaluate Performance
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- Evaluation Metrics on Test Set ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")


--- Evaluation Metrics on Test Set ---
Mean Squared Error (MSE): 250.73
Mean Absolute Error (MAE): 12.64


# ??????

In [25]:
import os
import torch
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =====================================================================
# 1. DATA LOADING FUNCTIONS
# =====================================================================
def load_pt_samples(input_dir, case_ids, seq_len):
    """Loads 3D sliding windows directly from .pt files for specific patients."""
    X_list = []
    Y_list = []

    for cid in case_ids:
        sample_path = os.path.join(input_dir, f'case_{cid}.pt')
        if not os.path.exists(sample_path):
            continue
            
        data = torch.load(sample_path, weights_only=False)
        x = data['features'].numpy()
        y = data['bis'].numpy()
            
        num_samples = x.shape[0]
        if num_samples <= seq_len:
            continue
            
        # Create sliding windows per patient
        X_case = np.array([x[i:i+seq_len] for i in range(num_samples - seq_len)])
        Y_case = np.array([y[i+seq_len] for i in range(num_samples - seq_len)]).reshape(-1, 1)
        
        X_list.append(X_case)
        Y_list.append(Y_case)

    # Combine all patients
    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)
    
    return X, Y

# =====================================================================
# 2. FEATURE ENGINEERING FUNCTION (3D to 2D)
# =====================================================================
def extract_xgboost_features(X_3d):
    """
    Converts (Samples, Seq_Len, Features) to (Samples, Features * 4)
    by calculating Mean, Std, Max, and Min across the time window.
    Using np.nan* functions to safely ignore any lingering missing values.
    """
    print(f"Extracting stats from 3D array of shape: {X_3d.shape}...")
    
    # Calculate statistics across the time dimension (axis=1)
    x_mean = np.nanmean(X_3d, axis=1)
    x_std  = np.nanstd(X_3d, axis=1)
    x_max  = np.nanmax(X_3d, axis=1)
    x_min  = np.nanmin(X_3d, axis=1)
    
    # Stack horizontally to create the final 2D feature matrix
    # If original features=19, output will be 19*4 = 76 features
    X_2d = np.hstack([x_mean, x_std, x_max, x_min])
    
    return X_2d

# =====================================================================
# 3. MAIN EXECUTION PIPELINE
# =====================================================================
# Configuration
INPUT_DIR = '../../data/processed/eeg'
CASES_FILE = '../../data/processed/cases_data.csv'
SEQ_LEN = 60 # 60-second window

# A. Load Patient IDs and Split
cases_master = pd.read_csv(CASES_FILE)
all_ids = cases_master['caseid'].tolist()

# Split by patient ID (80% Train, 20% Test)
train_ids, test_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
print(f"Training on {len(train_ids)} patients, Testing on {len(test_ids)} patients.\n")

# B. Load 3D Sequences
print("Loading Training Data...")
X_train_3d, Y_train = load_pt_samples(INPUT_DIR, train_ids, SEQ_LEN)

print("Loading Testing Data...")
X_test_3d, Y_test = load_pt_samples(INPUT_DIR, test_ids, SEQ_LEN)

# C. Convert to 2D Statistical Features
X_train_2d = extract_xgboost_features(X_train_3d)
X_test_2d  = extract_xgboost_features(X_test_3d)

print(f"\nFinal Training Matrix Shape: {X_train_2d.shape}")
print(f"Final Testing Matrix Shape:  {X_test_2d.shape}\n")

# C.5 CLEAN TARGET VARIABLES (Remove NaNs from Labels)
# Flatten Y to 1D to create a boolean mask (True if NOT a NaN)
valid_train_mask = ~np.isnan(Y_train).flatten()
valid_test_mask  = ~np.isnan(Y_test).flatten()

# Filter both X and Y using the mask
X_train_clean = X_train_2d[valid_train_mask]
Y_train_clean = Y_train[valid_train_mask]

X_test_clean = X_test_2d[valid_test_mask]
Y_test_clean = Y_test[valid_test_mask]

print(f"Dropped {len(Y_train) - len(Y_train_clean)} train samples due to missing BIS labels.")
print(f"Dropped {len(Y_test) - len(Y_test_clean)} test samples due to missing BIS labels.")
print(f"Clean Training Matrix Shape: {X_train_clean.shape}")

# D. Prepare XGBoost Data Structures (Using the clean data!)
dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
dtest  = xgb.DMatrix(X_test_clean,  label=Y_test_clean)

# E. Define Model Parameters
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1
}

# ---> W&B STEP 1: Initialize the experiment <---
wandb.init(
    project="eeg-bis-prediction", # Name of your project dashboard
    config=params,                # Automatically tracks your hyperparameters!
    name="xgb-baseline"           # Optional: give this specific run a name
)

# F. Train the Model
print("Training XGBoost Model...")
evals = [(dtrain, 'train'), (dtest, 'eval')]

xgb_model = xgb.train(
    params=params, 
    dtrain=dtrain, 
    num_boost_round=500, 
    evals=evals, 
    early_stopping_rounds=20,
    verbose_eval=50,
    # ---> W&B STEP 2: Add the callback so it tracks metrics live <---
    callbacks=[WandbCallback()] 
)

# G. Final Evaluation
predictions = xgb_model.predict(dtest)
mae = mean_absolute_error(Y_test_clean, predictions)
rmse = np.sqrt(mean_squared_error(Y_test_clean, predictions))

print("\n--- FINAL RESULTS ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} BIS points")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} BIS points")

# ---> W&B STEP 3: Log your final evaluation metrics <---
wandb.log({
    "final_test_mae": mae,
    "final_test_rmse": rmse
})

# ---> W&B STEP 4: Tell W&B the script is finished <---
wandb.finish()

Training on 324 patients, Testing on 82 patients.

Loading Training Data...
Loading Testing Data...
Extracting stats from 3D array of shape: (1488608, 60, 19)...


C:\Users\mikol\AppData\Local\Temp\ipykernel_23412\3238881011.py:55: RuntimeWarning: Mean of empty slice
  x_mean = np.nanmean(X_3d, axis=1)
c:\Users\mikol\anaconda3\envs\master_thesis\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\mikol\AppData\Local\Temp\ipykernel_23412\3238881011.py:57: RuntimeWarning: All-NaN slice encountered
  x_max  = np.nanmax(X_3d, axis=1)
C:\Users\mikol\AppData\Local\Temp\ipykernel_23412\3238881011.py:58: RuntimeWarning: All-NaN slice encountered
  x_min  = np.nanmin(X_3d, axis=1)


Extracting stats from 3D array of shape: (380561, 60, 19)...

Final Training Matrix Shape: (1488608, 76)
Final Testing Matrix Shape:  (380561, 76)

Dropped 1449 train samples due to missing BIS labels.
Dropped 359 test samples due to missing BIS labels.
Clean Training Matrix Shape: (1487159, 76)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\mikol\_netrc.
wandb: Currently logged in as: mik-nowacki (mik-nowacki-chalmers-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training XGBoost Model...
[0]	train-rmse:13.91612	eval-rmse:13.73059
[50]	train-rmse:6.59024	eval-rmse:6.98224
[100]	train-rmse:6.10509	eval-rmse:6.82926
[150]	train-rmse:5.77424	eval-rmse:6.77997
[200]	train-rmse:5.49691	eval-rmse:6.74648
[240]	train-rmse:5.31872	eval-rmse:6.73590


wandb: ERROR The nbformat package was not found. It is required to save notebook history.



--- FINAL RESULTS ---
Mean Absolute Error (MAE): 4.66 BIS points
Root Mean Squared Error (RMSE): 6.74 BIS points


best_iteration,▁
best_score,▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇█
eval-rmse,█▄▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
final_test_mae,▁
final_test_rmse,▁
train-rmse,█▅▅▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_iteration,220
best_score,6.73453
epoch,240
final_test_mae,4.65547
